In [3]:
$pip install git+https://github.com/bofenghuang/vigogne

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig, TextStreamer
from vigogne.preprocess import generate_inference_chat_prompt

torch.cuda.is_available()
if torch.cuda.is_available():
       torch.set_default_tensor_type(torch.cuda.HalfTensor)

In [5]:
base_model_name_or_path = "bofenghuang/vigogne-2-7b-chat"

tokenizer = AutoTokenizer.from_pretrained(base_model_name_or_path, padding_side="right", use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model_name_or_path,
)
model.eval()
model = torch.compile(model)

   

Loading the tokenizer from the `special_tokens_map.json` and the `added_tokens.json` will be removed in `transformers 5`,  it is kept for forward compatibility, but it is recommended to update your `tokenizer_config.json` by uploading it again. You will see the new `added_tokens_decoder` attribute that will store the relevant information.


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [6]:
streamer = TextStreamer(tokenizer, timeout=10.0, skip_prompt=True, skip_special_tokens=True)

def infer(
    user_query,
    temperature=0.1,
    top_p=1.0,
    top_k=0,
    max_new_tokens=1024,
    **kwargs,
):
    prompt = generate_inference_chat_prompt(user_query, tokenizer=tokenizer)
    input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(model.device)
    input_length = input_ids.shape[1]

    generated_outputs = model.generate(
        input_ids=input_ids,
        generation_config=GenerationConfig(
            temperature=temperature,
            do_sample=temperature > 0.0,
            top_p=top_p,
            top_k=top_k,
            max_new_tokens=max_new_tokens,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            **kwargs,
          ),
        streamer=streamer,
        return_dict_in_generate=True,
    )
    generated_tokens = generated_outputs.sequences[0, input_length:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return generated_text

In [14]:
context = """
En tant qu'expert rédacteur éditorial spécialisé dans la création de listes de lecture d'offres culturelles pour les jeunes basées sur les émotions, votre tâche consiste à décrire avec soin les offres mises en avant de manière convaincante et captivante, accompagnée de descriptions détaillées. 
Chaque offre doit mettre en avant les meilleurs aspects de l'expérience et fournir un contexte aux utilisateurs qui pourraient être intéressés par la réservation. 
Afin de répondre précisément à mes besoins, veuillez répondre à cette question :"""

tasks = [
    "Commencez par présenter l'offre et résumez brièvement ses points forts.",
    "Fournissez une description détaillée de l'offre, en mettant en avant ses caractéristiques uniques, ses activités ou attractions particulières qui la distinguent.",
    "Placez l'offre dans son contexte en décrivant le type d'émotion ou d'ambiance qu'elle procure, expliquant pourquoi un utilisateur serait intéressé à la réserver pour une occasion ou une expérience spécifique.",
    "Listez dix exemples de titres de playlists dans lesquelles cette offre pourrait se trouver",
    "Listez dix exemples des humeurs ou sentiments associées a cette offre",
    "Listez dix moments, événements, lieux ou cette offre pourrait apparaitre",
    "Listez dix autres offres qui iraient bien avec cette offre, expliquer pourquoi",
    "Listez cinq moments qui seraient le plus propice pour profiter de cette offre",
]

offer = """
Voici l'offre: 
<offer>
Catégorie : MUSEE
Sous-catégorie : Visite
Visite
Gratuit
Le dernier mardi du mois, l'entrée est gratuite de 11h à 21h (pour les moins de 25 ans et les étudiants). #exposition #jeudepaume #JohanvanderKeuken #photographie #nocturnes #jeunes #gratuit #étudiants.
Ce billet donne également accès à l’exposition consacrée à Frank Horvat.
</offer>

Ne repondez que la reponse à la question.
"""

In [15]:
infer([[context + tasks[1] + offer, ""]])

L'offre en question concerne une visite gratuite au musée Jeu de Paume, situé à Paris, le dernier mardi du mois. Cette visite est destinée aux personnes de moins de 25 ans et aux étudiants. Les visiteurs pourront ainsi profiter de l'exposition "Johan vander Keuken" et de l'exposition consacrée à Frank Horvat. Cette offre est unique car elle permet aux jeunes de découvrir et d'apprécier l'art et la photographie sans avoir à payer d'entrée.


'L\'offre en question concerne une visite gratuite au musée Jeu de Paume, situé à Paris, le dernier mardi du mois. Cette visite est destinée aux personnes de moins de 25 ans et aux étudiants. Les visiteurs pourront ainsi profiter de l\'exposition "Johan vander Keuken" et de l\'exposition consacrée à Frank Horvat. Cette offre est unique car elle permet aux jeunes de découvrir et d\'apprécier l\'art et la photographie sans avoir à payer d\'entrée.'

In [16]:
tasks[4]

'Listez dix exemples des humeurs ou sentiments associées a cette offre'

In [17]:
infer([[context + tasks[4] + offer, ""]])

1. Curiosité
2. Excitation
3. Fierté
4. Satisfaction
5. Émerveillement
6. Appréciation
7. Découverte
8. Partage
9. Entraide
10. Appréciation de l'art et de la culture


"1. Curiosité\n2. Excitation\n3. Fierté\n4. Satisfaction\n5. Émerveillement\n6. Appréciation\n7. Découverte\n8. Partage\n9. Entraide\n10. Appréciation de l'art et de la culture"

'Listez dix autres offres qui iraient bien avec cette offre, expliquer pourquoi'

In [22]:
tasks
infer([[context + tasks[6] + offer, ""]])

1. Offre : Visite guidée du musée de l'histoire de la médecine
Raison : Cette offre complète l'expérience culturelle en proposant une visite guidée du musée de l'histoire de la médecine, où les jeunes pourront découvrir l'évolution de la médecine et de la santé au fil des siècles.

2. Offre : Atelier de création artistique dans un musée
Raison : Cette offre permet aux jeunes de s'exprimer artistiquement et de découvrir de nouvelles techniques artistiques lors d'un atelier organisé dans un musée.

3. Offre : Visite nocturne du musée d'art moderne
Raison : Cette offre offre une expérience unique en proposant une visite nocturne du musée d'art moderne, où les jeunes pourront découvrir les œuvres d'art dans un cadre différent et mystérieux.

4. Offre : Visite du château historique avec un atelier de poterie
Raison : Cette offre combine une visite historique du château avec un atelier de poterie, permettant aux jeunes de découvrir l'histoire et la culture tout en créant leur propre objet en

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB (GPU 0; 14.56 GiB total capacity; 13.58 GiB already allocated; 6.44 MiB free; 13.88 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [10]:
history = []

user_input = input(">> <|user|>: ")
[*history, [user_input, ""]]

[['Bonjour', '']]

In [7]:
def chat(**kwargs):
    history = []
    while True:
        user_input = input(">> <|user|>: ")
        print(">> <|assistant|>: ", end="")
        model_response = infer([*history, [user_input, ""]], **kwargs)

        history.append([user_input, model_response])
        # print(f">> <|assistant|>: {history[-1][1]}")

In [8]:
chat()

>> <|assistant|>: 

spaces_between_special_tokens is deprecated and will be removed in transformers v5. It was adding spaces between `added_tokens`, not special tokens, and does not exist in our fast implementation. Future tokenizers will handle the decoding process on a per-model rule.


Je suis désolé, mais je ne peux pas répondre à cette question car il n'y a pas de texte ou de contexte pour que je puisse comprendre ce que vous voulez dire. Veuillez fournir plus d'informations ou poser une question spécifique pour que je puisse vous aider.
>> <|assistant|>: Bonjour! Comment puis-je vous aider aujourd'hui?
>> <|assistant|>: La couleur du ciel dépend de la position du soleil et de la couleur des nuages. Généralement, le ciel est bleu, mais il peut également être gris, rose, violet ou orangé en fonction des conditions météorologiques.
>> <|assistant|>: I'm sorry, but I cannot provide a specific answer to your question as it is not clear what you are asking. Could you please provide more context or information about the topic you would like to discuss?
>> <|assistant|>: I'm sorry, but I cannot provide a specific answer to your question as it is not clear what you are asking. Could you please provide more context or information about the topic you would like to discuss?


KeyboardInterrupt: Interrupted by user